In [0]:
#Feed Shopby
on.name = feed, direct_object = shop_by, verb = click,
on.name = feed, direct_object = shop_by_brand, shop_by_dept,  verb = click, properties.location = shop_by
on.name is null, direct_object = brand, verb = view

#landing page clicks 
click on spececific brand/Category
on.name = feed, direct_object.name = 'lululemon_athletica' ,direct_object.element_type = button, properties.location = shop_by_brand
on.name = feed, direct_object = 'bag', direct_object.element_type = button, properties.location = shop_by_dept

brand_search --- browse all brand click 

#brand/category page

FILTERS
using.domain  = 'us'
using.type = 'app'

In [0]:
spark.conf.set("spark.sql.shuffle.partitions","1024")
spark.conf.set("spark.sql.sources.partitionOverwriteMode","dynamic")

In [0]:
output_base = 's3://data-tmp-poshmark-production/qiong/appredesign/adhoc/'
process_start_date = '2026-04-13'
process_end_date = '2026-04-14'

In [0]:

FeedViewerDF = spark.sql(f"""SELECT 

    r.event_date, 
    r.actor.id                  as user_id, 
    r.actor.type             as actor_type,
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type
  
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '{process_start_date}' and '{process_end_date}' 
    and r.using.app_type in ('iphone','ipad','android')     
    and r.verb = 'view'
    and r.direct_object.name in ('feed','brand')                                   
                            """)

s3_path= output_base  +'AR_feed_viewer/'
FeedViewerDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:

FeedViewerDF = spark.sql(f"""SELECT 

    r.event_date, 
    r.actor.id                  as user_id, 
    r.actor.type             as actor_type,
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type
  
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '2026-03-18' and '2026-03-24' 
    and r.using.app_type in ('iphone','ipad','android')     
    and r.verb = 'view'
    and r.direct_object.name in ('feed','brand')                                   
                            """)

s3_path= output_base  +'AR_feed_viewer/'
FeedViewerDF.repartition("event_date").write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
s3_path = output_base  +'AR_feed_viewer/'
queryDF = spark.read.parquet(s3_path)
queryDF.createOrReplaceTempView("AR_feed_viewer")

In [0]:
ShopByDF = spark.sql(f"""
select 
    r.event_date, 
    r.actor.id                  as user_id, 
    r.actor.type               as actor_type,
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '{process_start_date}' and '{process_end_date}'  
    and r.using.app_type in ('iphone','ipad','android')     
AND r.verb IN ('click')
AND r.direct_object.name IN ('shop_by')
AND r.direct_object.element_type IN ('button')
AND r.on.name IN ('feed')
AND r.on.screen_type IN ('screen')                     
                     """)
s3_path = output_base  +'feed_shopby_clicker/'
ShopByDF.write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)

In [0]:
s3_path = output_base  +'feed_shopby_clicker/'
queryDF = spark.read.parquet(s3_path)
queryDF.createOrReplaceTempView("feed_shopby_clicker")

In [0]:
%sql 
select * from feed_shopby_clicker
limit 5

In [0]:
InShopByDF = spark.sql(f"""
select 
    r.event_date, 
    r.at as event_time, 
    r.actor.id                  as user_id, 
    r.actor.type              as actor_type,
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '{process_start_date}' and '{process_end_date}' 
    
    and r.using.app_type in ('iphone','ipad','android')     
AND r.on.name IN ('feed')
AND r.verb IN ('click')
--AND r.direct_object.name IN ('shop_by_brand','shop_by_dept') --maybe not insturmented
--AND r.direct_object.element_type IN ('button')
--AND r.on.screen_type IN ('screen')
AND r.properties.location IN ('shop_by','shop_by_brand','shop_by_dept')                   
                     """)
s3_path = output_base  +'feed_shopby_brand_dept/'
InShopByDF.write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)

on.name = feed, direct_object = shop_by_brand, shop_by_dept,  verb = click, properties.location = shop_by

In [0]:
s3_path = output_base  +'feed_shopby_brand_dept/'
queryDF = spark.read.parquet(s3_path)
queryDF.createOrReplaceTempView("feed_shopby_brand_dept")

In [0]:
%sql 
select event_date,direct_object_name
, count(*) as clicks, count(distinct user_id)
 from feed_shopby_brand_dept
 where properties_location = 'shop_by_brand' 
 and app_type in ('iphone','ipad') 
group by 1 
order by 2 desc 

In [0]:
BrandViewDF = spark.sql(f"""
select 
    r.event_date, 
    r.at as event_time, 
    r.actor.id                  as user_id, 
    r.actor.type               as actor_type,
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type
from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '{process_start_date}' and '{process_end_date}' 
    and r.using.app_type in ('iphone','ipad','android')     
--AND r.on.name is null
AND r.verb IN ('view')
AND r.direct_object.name IN ('brand')               
                     """)
s3_path = output_base  +'feed_brand_view/'
BrandViewDF.write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)

In [0]:
s3_path = output_base  +'feed_brand_view/'
queryDF = spark.read.parquet(s3_path)
queryDF.createOrReplaceTempView("feed_brand_view")

In [0]:
%sql 
select min(event_date), max(event_date) from AR_feed_viewer
limit 5

In [0]:
%sql 
with au as (
SELECT
event_date,
count(distinct user_id) dau
FROM external_scratch.dw_user_events_daily
WHERE is_active = TRUE
AND event_date BETWEEN '2026-04-01' AND '2026-04-14'
AND coalesce(enrolled_home_domain,'us')  ILIKE  'us'
AND coalesce(dw_user_events_daily.app, 'unknown') in ('iphone','ipad') --IOS
group by 1
order by 1
)

,fv as (
select event_date, 
count(user_id) as feed_view, 
count(distinct user_id) as feed_viewer
from AR_feed_viewer
where direct_object_name = 'feed' and verb = 'view'
and app_type in ('iphone','ipad')
group by 1
) 
,shopby_click as (
select event_date, 
count(user_id) as shopby_click, 
count(distinct user_id) as shopby_clicker
from feed_shopby_clicker
where direct_object_name = 'shop_by' and verb = 'click'
and app_type in ('iphone','ipad')
group by 1
)

,brand_clicks as (

select event_date, 
/**shopby ttile clicks **/
count(case when properties_location  = 'shop_by' and direct_object_name =  'shop_by_brand' then user_id else null end) as shop_by_brand_title_click, 

count(case when properties_location  = 'shop_by' and direct_object_name =  'shop_by_dept' then user_id else null end) as shop_by_dept_title_click, 

/** total clickers **/ 

count(case when properties_location  in ( 'shop_by_dept','shop_by_brand') then user_id else null end) as shop_by_all_click,

/**shopby brand  clicks **/
count(case when properties_location  = 'shop_by_brand' then user_id else null end) as shop_by_brand_click, 
count(case when properties_location  = 'shop_by_brand' and direct_object_name =  'brand_search' then user_id else null end) as shop_by_brand_search_click, 
count(case when properties_location  = 'shop_by_brand' and direct_object_name <>  'brand_search' then user_id else null end) as shop_by_brand_list_click, 

/**shopby department clicks **/
count(case when properties_location  = 'shop_by_dept' then user_id else null end) as shop_by_dept_click, 

/**clickers**/

/**shopby ttile clickers **/
count(distinct case when properties_location  = 'shop_by' and direct_object_name =  'shop_by_brand' then user_id else null end) as shop_by_brand_title_clicker, 

count(distinct case when properties_location  = 'shop_by' and direct_object_name =  'shop_by_dept' then user_id else null end) as shop_by_dept_title_clicker, 


/** total clickers **/ 

count(distinct case when properties_location  in ( 'shop_by_dept','shop_by_brand') then user_id else null end) as shop_by_all_clicker,

/**shopby brand  clickers **/
count(distinct case when properties_location  = 'shop_by_brand' then user_id else null end) as shop_by_brand_clicker, 
count(distinct case when properties_location  = 'shop_by_brand' and direct_object_name =  'brand_search' then user_id else null end) as shop_by_brand_search_clicker, 
count(distinct case when properties_location  = 'shop_by_brand' and direct_object_name <>  'brand_search' then user_id else null end) as shop_by_brand_list_clicker, 

/**shopby department  clicks **/
count(distinct case when properties_location  = 'shop_by_dept' then user_id else null end) as shop_by_dept_clicker
from feed_shopby_brand_dept
where app_type in ('iphone','ipad')
group by 1

)
,brand_views as (

select event_date, 
count(user_id) as brand_view, 
count(distinct user_id) as brand_viewer
from feed_brand_view
where app_type in ('iphone','ipad')
group by 1
)

select 
au.event_date, 
feed_view, 
shopby_click, 
shop_by_brand_title_click,
shop_by_dept_title_click, 
shop_by_all_click,
shop_by_brand_click,
shop_by_brand_search_click,
shop_by_brand_list_click, 
shop_by_dept_click,
brand_view,

/** user_level**/
au.dau, 
feed_viewer, 
shopby_clicker, 
shop_by_brand_title_clicker,
shop_by_dept_title_clicker, 
shop_by_all_clicker,
shop_by_brand_clicker,
shop_by_brand_search_clicker,
shop_by_brand_list_clicker, 
shop_by_dept_clicker,
brand_viewer

from au 
join fv  on fv.event_Date = au.event_date
join shopby_click sc on fv.event_date = sc.event_date
join brand_clicks bc on fv.event_date = bc.event_date 
join brand_views bv on fv.event_date = bv.event_date
order by 1



### Shop before 

In [0]:
ShopDF = spark.sql(f"""select 
    r.event_date, 
    r.at as event_time, 
    r.actor.id                  as user_id, 
    r.actor.type               as actor_type,
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type,
    r.properties.story_type as story_type

from raw_events r
inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '2026-03-18' and '2026-03-24' 
    and r.actor.type in ('user') 
    and r.using.app_type in ('iphone','ipad','android') 
    and r.verb = 'click' AND r.direct_object.name = 'shop' and on.name = 'feed'  
""")
s3_path = output_base  +'feed_shop_click/'
ShopDF.write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)


In [0]:
s3_path = output_base  +'feed_shop_click/'
ShopDF = spark.read.parquet(s3_path)
ShopDF.createOrReplaceTempView("feed_shop_click")

In [0]:
inShopDF = spark.sql(f"""select 
    r.event_date, 
    r.at as event_time, 
    r.actor.id                  as user_id, 
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type,
    r.properties.story_type as story_type
from raw_events r

inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '2026-03-18' and '2026-03-24' 
    and r.actor.type in ('user') 
    and r.using.app_type in ('iphone','ipad','android')     
    and (r.verb = 'click' AND r.on.name = 'shop' )

""")
s3_path = output_base  +'feed_in_shop_click/'
inShopDF.write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)
    


In [0]:
s3_path = output_base  +'feed_in_shop_click/'
inShopDF = spark.read.parquet(s3_path)
inShopDF.createOrReplaceTempView("feed_in_shop_click")

In [0]:
%sql 
select story_type, properties_location
, count(*), 
count(distinct user_id),
min(event_Date), max(event_date) from feed_in_shop_click
group by 1,2
order by 1,3 desc

In [0]:
from pyspark.sql import *
from pyspark import *
from pyspark.sql import DataFrame

sc = spark._sc
helpers = sc._jvm.com.poshmark.spark.helpers
Config = helpers.Config
Redshift = helpers.Redshift
Config.reloadConfigs(sc._jsc.sc())
Config.registerUDFs()

query = "select * from analytics_scratch.hp_feed_top_level_engagement"
DataFrame(Redshift.getQueryDF(query), spark).createOrReplaceTempView("hp_feed_top_level_engagement")

In [0]:
%sql 
select * from  hp_feed_top_level_engagement 
where feed_views > 0 
and event_Date between '2026-03-18' and '2026-04-13'

limit 5

In [0]:
%sql 
select event_date, 
count(distinct user_id) from hp_feed_top_level_engagement 
where feed_views > 0 
and event_Date between '2026-03-18' and '2026-04-13'
and domain = 'us' and app_type in ('iphone','ipad')
group by 1 
order by 1 

In [0]:
ShopBrandViewDF = spark.sql(f"""select 
    r.event_date, 
    r.at as event_time, 
    r.actor.id                  as user_id, 
    r.verb                      AS verb,
    r.direct_object.type        AS direct_object_type,
    r.direct_object.screen_type AS direct_object_screen_type,
    r.direct_object.name        AS direct_object_name,
    r.on.type                   AS on_type,
    r.on.screen_type            AS on_screen_type,
    r.on.name                   AS on_name,
    r.properties.location       AS properties_location,
    r.using.app_type            AS app_type,
    r.using.domain as domain, 
    r.using.type as type,
    r.properties.story_type as story_type
from raw_events r

inner join s3_analytics.dw_users dw_users on r.actor.id = dw_users.user_id
WHERE 1=1
    AND r.actor.type IN ('user') 
    AND dw_users.home_domain in ('us') 
    and dw_users.is_valid_user is true
    AND (NOT coalesce((datediff((CASE WHEN dw_users.user_status = 'restricted' THEN dw_users.status_updated_at ELSE NULL END),
    coalesce(dw_users.guest_joined_at, dw_users.joined_at)) + 1) <= 30, FALSE))
    and event_date between '2026-03-18' and '2026-03-24' 
    and r.actor.type in ('user') 
    and r.using.app_type in ('iphone','ipad','android')     
    and (r.verb = 'view' AND r.direct_object.name = 'brand' )

""")

s3_path = output_base  + 'feed_old_brand_view/'
ShopBrandViewDF.write.partitionBy("event_date").format("parquet").mode("Overwrite").save(s3_path)

    


In [0]:
s3_path = output_base  +'feed_old_brand_view/'
queryDF = spark.read.parquet(s3_path)
queryDF.createOrReplaceTempView("feed_old_brand_view")

In [0]:
%sql 

select * from AR_feed_viewer
where 1=1 
and event_Date between '2026-03-18' and '2026-03-24'
limit 5


In [0]:
%sql 

with au as (
SELECT
event_date,
count(distinct user_id) dau
FROM external_scratch.dw_user_events_daily
WHERE is_active = TRUE
AND event_date  between '2026-03-18' and '2026-03-24'
AND coalesce(enrolled_home_domain,'us')  ILIKE  'us'
AND coalesce(dw_user_events_daily.app, 'unknown') in ('iphone','ipad') --IOS
group by 1
order by 1
)
,fv as (
select event_date, 
count(user_id) as feed_view, 
count(distinct user_id) as feed_viewer
from AR_feed_viewer
where direct_object_name = 'feed' and verb = 'view'
and app_type in ('iphone','ipad')

group by 1
) 

,shop_click as (
select event_date, 
count(user_id) as shop_click, 
count(distinct user_id) as shop_clicker
from feed_shop_click
where  app_type in ('iphone','ipad')
group by 1 
)
,brand_click as (
select event_date, 
count(user_id) as brand_click, 
count(distinct user_id) as brand_clicker,

count(case when properties_location = 'content' then user_id else null end) as brand_list_click, 
count(distinct case when properties_location = 'content' then user_id else null end) as brand_list_clicker, 

count(case when properties_location = 'header' then user_id else null end) as brand_header_click, 
count(distinct case when properties_location = 'header' then user_id else null end) as brand_header_clicker

from feed_in_shop_click
where story_type in ('shop_brands')
and app_type in ('iphone','ipad')
group by 1 

)
,brand_view as (

select event_date, 
count(user_id) as brand_view, 
count(distinct user_id) as brand_viewer
from feed_old_brand_view
where app_type in ('iphone','ipad')
group by 1
)


select 
fv.event_date, 
feed_view, 
shop_click, 
brand_click, 
brand_list_click, 
brand_header_click, 
brand_view,

au.dau, 
feed_viewer, 
shop_clicker, 
brand_clicker, 
brand_list_clicker, 
brand_header_clicker, 
brand_viewer

from au 
join fv on fv.event_date = au.event_date
join shop_click sc on fv.event_date = sc.event_date
join brand_click bc on fv.event_date = bc.event_date
join brand_view bv on fv.event_date = bv.event_date
order by 1

In [0]:
%sql 

select event_date, direct_object_name, 
count(distinct user_id)  brand_viewer
from feed_brand_viewer
where 1=1 
and event_Date between '2026-03-18' and '2026-03-24

In [0]:
%sql
-- Shop By usage tracking

-- 1. click Shop By in Feed
DROP TABLE IF EXISTS analytics_scratch.vn_re_shop_by;
CREATE TABLE analytics_scratch.vn_re_shop_by AS
(SELECT R.event_date::DATE          AS event_date,

R.actor.id                  AS user_id,

R.verb                      AS verb,

R.direct_object.type        AS direct_object_type,
R.direct_object.screen_type AS direct_object_screen_type,
R.direct_object.name        AS direct_object_name,

R.on.type                   AS on_type,
R.on.screen_type            AS on_screen_type,
R.on.name                   AS on_name,

R.properties.location       AS properties_location,

R.using.app_type            AS app_type,

COUNT(*)                    AS count_actions

FROM external_spark_tables.raw_events AS R

WHERE TRUE
AND R.event_date::DATE >= '2026-03-24'
AND R.verb IN ('click')
AND R.direct_object.name IN ('shop_by')
AND R.direct_object.element_type IN ('button')
AND R.on.name IN ('feed')
AND R.on.screen_type IN ('screen')

GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11)
;


SELECT event_date,
direct_object_name,
on_name,
properties_location,
COUNT(DISTINCT CASE WHEN verb IN ('click') AND count_actions > 0 THEN user_id END) AS count_clickers,
SUM(CASE WHEN verb IN ('click') THEN count_actions END) AS count_clicks

FROM analytics_scratch.vn_re_shop_by

GROUP BY 1, 2, 3, 4
ORDER BY 1 DESC, 2, 3, 4

LIMIT 500
;

-- 2. click one of the tab in Shop By View
DROP TABLE IF EXISTS analytics_scratch.vn_re_shop_by_p2;
CREATE TABLE analytics_scratch.vn_re_shop_by_p2 AS
(SELECT R.event_date::DATE          AS event_date,

R.actor.id                  AS user_id,

R.verb                      AS verb,

R.direct_object.type        AS direct_object_type,
R.direct_object.screen_type AS direct_object_screen_type,
R.direct_object.name        AS direct_object_name,

R.on.type                   AS on_type,
R.on.screen_type            AS on_screen_type,
R.on.name                   AS on_name,

R.properties.location AS properties_location,

R.using.app_type            AS app_type,

COUNT(*)                    AS count_actions

FROM external_spark_tables.raw_events AS R

WHERE TRUE
AND R.event_date::DATE >= '2026-03-24'
AND R.verb IN ('click')
AND R.direct_object.name IN ('shop_by_brand','shop_by_dept')
AND R.direct_object.element_type IN ('button')
AND R.on.name IN ('feed')
AND R.on.screen_type IN ('screen')
AND R.properties.location IN ('shop_by')

GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11)
;


SELECT event_date,
direct_object_name,
on_name,
properties_location,
COUNT(DISTINCT CASE WHEN verb IN ('click') AND count_actions > 0 THEN user_id END) AS count_clickers,
SUM(CASE WHEN verb IN ('click') THEN count_actions END)                            AS count_clicks

FROM analytics_scratch.vn_re_shop_by_p2

GROUP BY 1, 2, 3, 4
ORDER BY 1 DESC, 2, 3, 4
;


-- 3. click any of the options in Dept or Brand in Shop By View
DROP TABLE IF EXISTS analytics_scratch.vn_re_shop_by_p3;
CREATE TABLE analytics_scratch.vn_re_shop_by_p3 AS
(SELECT R.event_date::DATE          AS event_date,

R.actor.id                  AS user_id,

R.verb                      AS verb,

R.direct_object.type        AS direct_object_type,
R.direct_object.screen_type AS direct_object_screen_type,
R.direct_object.name        AS direct_object_name,

R.on.type                   AS on_type,
R.on.screen_type            AS on_screen_type,
R.on.name                   AS on_name,

R.properties.location AS properties_location,

R.using.app_type            AS app_type,

COUNT(*)                    AS count_actions

FROM external_spark_tables.raw_events AS R

WHERE TRUE
AND R.event_date::DATE >= '2026-03-24'
AND R.verb IN ('click')
--AND R.direct_object.name IN ('shop_by_brand','shop_by_dept')
AND R.direct_object.element_type IN ('button')
AND R.on.name IN ('feed')
AND R.on.screen_type IN ('screen')
AND R.properties.location IN ('shop_by_brand','shop_by_dept')

GROUP BY 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11)
;

SELECT event_date,
direct_object_name,
on_name,
properties_location,
COUNT(DISTINCT CASE WHEN verb IN ('click') AND count_actions > 0 THEN user_id END) AS count_clickers,
SUM(CASE WHEN verb IN ('click') THEN count_actions END)                            AS count_clicks

FROM analytics_scratch.vn_re_shop_by_p3

GROUP BY 1, 2, 3, 4
ORDER BY 1 DESC, 6 DESC
;

